In [26]:
from langchain_core.documents import Document

In [27]:
doc = Document(
    page_content = "this is the main content i am using to create a RAG Applications",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author": "yash",
        "data_created": "2026-05-17"
        

    }
)

In [28]:
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'yash', 'data_created': '2026-05-17'}, page_content='this is the main content i am using to create a RAG Applications')

In [29]:
## create a simple txt file 
import os 
os.makedirs("../data/txt_files",exist_ok = True)

In [30]:
sample_texts={
    "../data/txt_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/txt_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


In [31]:
## text leader 
from langchain_community.document_loaders   import TextLoader

## how to read 
loader= TextLoader("../data/txt_files/python_intro.txt",encoding="utf-8")
loader

In [32]:
document = loader.load()
print(document)

[Document(metadata={'source': '../data/txt_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [33]:
## Directory loader 

In [34]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/txt_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\txt_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\txt_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popula

In [35]:
## pdf loader 
### Directory Loader
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'producer': 'jsPDF 3.0.1', 'creator': '', 'creationdate': '2025-12-10T22:58:34+05:30', 'source': '..\\data\\pdf\\agreement.pdf', 'file_path': '..\\data\\pdf\\agreement.pdf', 'total_pages': 4, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20251210225834+05'30'", 'page': 0}, page_content='CAR LEASE AGREEMENT\n(Motor Vehicle Lease Contract – Synthetic, Legally Styled)\nAgreement Number: CLA-2025-44812\nDate of Agreement: February 15, 2025\n1. PARTIES\nLessor (Leasing Company):\nPrimeDrive Auto Leasing LLC\n2140 Westwood Business Park\nChicago, IL 60616\nPhone: (312) 555-9283\nLessee (Customer):\nJakub Nowak\nul. Kwiatowa 23/4\nWarsaw, Poland 02-658\nPhone: +48 600 854 739\n2. VEHICLE INFORMATION\nField        \nDetails\nVIN        \nNMTBE3JE70R148437\nBrand        \nToyota\nModel        \nCorolla\nTrim/Version        \n1.6 Premium\nBody Type        \nSedan\nFuel Type       

In [36]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [42]:
## pdf loader possible variable is docuements 

In [43]:
chunks=split_documents(documents)
chunks

Split 9 documents into 12 chunks

Example chunk:
Content: CAR LEASE AGREEMENT
(Motor Vehicle Lease Contract – Synthetic, Legally Styled)
Agreement Number: CLA-2025-44812
Date of Agreement: February 15, 2025
1. PARTIES
Lessor (Leasing Company):
PrimeDrive Aut...
Metadata: {'producer': 'jsPDF 3.0.1', 'creator': '', 'creationdate': '2025-12-10T22:58:34+05:30', 'source': '..\\data\\pdf\\agreement.pdf', 'file_path': '..\\data\\pdf\\agreement.pdf', 'total_pages': 4, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20251210225834+05'30'", 'page': 0}


[Document(metadata={'producer': 'jsPDF 3.0.1', 'creator': '', 'creationdate': '2025-12-10T22:58:34+05:30', 'source': '..\\data\\pdf\\agreement.pdf', 'file_path': '..\\data\\pdf\\agreement.pdf', 'total_pages': 4, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20251210225834+05'30'", 'page': 0}, page_content='CAR LEASE AGREEMENT\n(Motor Vehicle Lease Contract – Synthetic, Legally Styled)\nAgreement Number: CLA-2025-44812\nDate of Agreement: February 15, 2025\n1. PARTIES\nLessor (Leasing Company):\nPrimeDrive Auto Leasing LLC\n2140 Westwood Business Park\nChicago, IL 60616\nPhone: (312) 555-9283\nLessee (Customer):\nJakub Nowak\nul. Kwiatowa 23/4\nWarsaw, Poland 02-658\nPhone: +48 600 854 739\n2. VEHICLE INFORMATION\nField        \nDetails\nVIN        \nNMTBE3JE70R148437\nBrand        \nToyota\nModel        \nCorolla\nTrim/Version        \n1.6 Premium\nBody Type        \nSedan\nFuel Type       

#### Embedding And VectorStoredb

In [37]:
import numpy as np
from sentence_transformers import SentenceTransformer ## embeddings 
import chromadb
from chromadb.config import Settings
import uuid # data id store 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5821.58it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Admin pc\AppData\Local\Temp\ipykernel_7144\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


#### Vector Store


In [39]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store ## store data for hardisk
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() ## function _initialize_store - also initialize also a vector store 


    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
## add documents 
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
            


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [44]:
chunks

[Document(metadata={'producer': 'jsPDF 3.0.1', 'creator': '', 'creationdate': '2025-12-10T22:58:34+05:30', 'source': '..\\data\\pdf\\agreement.pdf', 'file_path': '..\\data\\pdf\\agreement.pdf', 'total_pages': 4, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20251210225834+05'30'", 'page': 0}, page_content='CAR LEASE AGREEMENT\n(Motor Vehicle Lease Contract – Synthetic, Legally Styled)\nAgreement Number: CLA-2025-44812\nDate of Agreement: February 15, 2025\n1. PARTIES\nLessor (Leasing Company):\nPrimeDrive Auto Leasing LLC\n2140 Westwood Business Park\nChicago, IL 60616\nPhone: (312) 555-9283\nLessee (Customer):\nJakub Nowak\nul. Kwiatowa 23/4\nWarsaw, Poland 02-658\nPhone: +48 600 854 739\n2. VEHICLE INFORMATION\nField        \nDetails\nVIN        \nNMTBE3JE70R148437\nBrand        \nToyota\nModel        \nCorolla\nTrim/Version        \n1.6 Premium\nBody Type        \nSedan\nFuel Type       

In [45]:
### text to embeddings .


In [47]:
texts = [doc.page_content for doc in chunks]
texts

['CAR LEASE AGREEMENT\n(Motor Vehicle Lease Contract – Synthetic, Legally Styled)\nAgreement Number: CLA-2025-44812\nDate of Agreement: February 15, 2025\n1. PARTIES\nLessor (Leasing Company):\nPrimeDrive Auto Leasing LLC\n2140 Westwood Business Park\nChicago, IL 60616\nPhone: (312) 555-9283\nLessee (Customer):\nJakub Nowak\nul. Kwiatowa 23/4\nWarsaw, Poland 02-658\nPhone: +48 600 854 739\n2. VEHICLE INFORMATION\nField        \nDetails\nVIN        \nNMTBE3JE70R148437\nBrand        \nToyota\nModel        \nCorolla\nTrim/Version        \n1.6 Premium\nBody Type        \nSedan\nFuel Type        \nGasoline\nColor        \nSilver\nYear of Manufacture        \n2015\nCountry of Registration        \nPoland\nCurrent Mileage        \n150,000–200,000 km\nThe vehicle is leased in good mechanical condition with expected wear for its mileage and \nage.\n3. LEASE TERM\nItem        \nValue\nLease Duration        \n36 months\nStart Date        \nMarch 1, 2025\nEnd Date        \nFebruary 28, 2028\nGener

In [48]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s]

Generated embeddings with shape: (12, 384)
Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents in collection: 12


### Retriever Pipeline From VectorStore